In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OCQ_E.xpt"
csv_file = DATA_DIR / "OCQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source for final fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 OCQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OCD150",
    "OCQ180",
    "OCQ210",
    "OCD231",
    "OCD241",
    "OCQ260",
    "OCQ265",
    "OCD270",
    "OCQ290G",
    "OCQ290Q",
    "OCQ380",
    "OCD390G",
    "OCD391",
    "OCD392",
    "OCD395",
    "OCQ510",
    "OCQ520",
    "OCQ530",
    "OCQ540",
    "OCQ550",
    "OCQ560",
    "OCQ570",
    "OCQ580"
]


if df.shape != (6546, 24):
    raise ValueError(
        f"Unexpected OCQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "OCQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: OCQ_E structure = "
    "6,546 rows x 24 columns."
)


# ============================================================
# 4. CDC CHECK ITEMS
# ============================================================

#
# CDC HTML contains:
#
# OCQ200 - CHECK ITEM
# OCQ500 - CHECK ITEM
#
# They are routing instructions and are NOT released
# variables in OCQ_E.xpt.
#
# DO NOT add them to CSV.
#

CHECK_ITEMS = [
    "OCQ200",
    "OCQ500"
]


for col in CHECK_ITEMS:
    if col in df.columns:
        raise ValueError(
            f"Unexpected CHECK ITEM in XPT: {col}"
        )


print(
    "PASS: OCQ200 and OCQ500 correctly absent "
    "from released XPT."
)


# ============================================================
# 5. NUMERIC FIELD CHECK
# ============================================================

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


if numeric_cols != EXPECTED_COLUMNS:
    raise ValueError(
        "Unexpected nonnumeric OCQ_E field found."
    )


print(
    "PASS: All 24 released fields are numeric."
)


# ============================================================
# 6. KNOWN SAS/XPORT TINY ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. VALIDATE EXACT TINY PATTERN
# ============================================================

EXPECTED_TINY_COUNTS = {
    "OCD270": 80,
    "OCQ290Q": 34,
    "OCD395": 15,
    "OCQ520": 147,
    "OCQ540": 99,
    "OCQ560": 82,
    "OCQ580": 142
}


EXPECTED_TINY_TOTAL = 599


actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != EXPECTED_TINY_TOTAL:
    raise ValueError(
        "Unexpected total tiny-artifact count."
    )


if true_zero_total != 0:
    raise ValueError(
        "Unexpected true zeros already decoded "
        "in original XPT."
    )


print(
    "\nPASS: Exact tiny pattern confirmed."
)


# ============================================================
# 8. APPROVED ZERO VARIABLES
# ============================================================

#
# CDC supports zero in exactly these affected fields:
#
# OCD270   = less than 1 month
# OCQ290Q  = less than 1 hour/day
# OCD395   = less than 1 month
# OCQ520   = less than 1 year
# OCQ540   = less than 1 year
# OCQ560   = less than 1 year
# OCQ580   = less than 1 year
#

APPROVED_ZERO_COLUMNS = set(
    EXPECTED_TINY_COUNTS
)


if (
    set(actual_tiny_counts)
    !=
    APPROVED_ZERO_COLUMNS
):
    raise ValueError(
        "STOP: Tiny values found in "
        "unexpected OCQ_E fields."
    )


print(
    "PASS: Tiny values occur only in "
    "CDC zero-permitted fields."
)


# ============================================================
# 9. RESTORE VALIDATED ZEROS
# ============================================================

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, expected_count in (
    EXPECTED_TINY_COUNTS.items()
):

    mask = df[col].eq(
        TINY_VALUE
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:
        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"tiny values, found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


print(
    "PASS: All 599 validated zeros restored."
)


# ============================================================
# 10. EXACT FINAL ZERO COUNTS
# ============================================================

for col, expected_count in (
    EXPECTED_TINY_COUNTS.items()
):

    actual_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:
        raise ValueError(
            f"{col}: final zero count differs."
        )


final_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


if final_zero_total != 599:
    raise ValueError(
        "Unexpected zeros introduced elsewhere."
    )


print(
    "PASS: Final zero counts exact."
)

print(
    "PASS: No unexpected zeros introduced."
)


# ============================================================
# 11. VERIFY NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:
    raise ValueError(
        "Tiny XPORT artifact remains."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 12. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:
    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:
    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):
    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 13. FRACTIONAL / DECIMAL AUDIT
# ============================================================

#
# After the validated tiny -> zero corrections,
# the actual attached OCQ_E contains NO genuine
# fractional values.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:
        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:
    raise ValueError(
        "STOP: Unexpected genuine decimals found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 14. CDC CHECK - OCD270
# ============================================================

#
# Number of months at main job
#
# 1-720  = 3,389
# 0      = 80
# 77777  = 2
# 99999  = 0
# Missing = 3,075
#

s = df["OCD270"]


if int(
    s.between(
        1,
        720
    ).sum()
) != 3389:
    raise ValueError(
        "OCD270 1-720 count differs."
    )


if int(
    s.eq(0).sum()
) != 80:
    raise ValueError(
        "OCD270 zero count differs."
    )


if int(
    s.eq(77777).sum()
) != 2:
    raise ValueError(
        "OCD270=77777 count differs."
    )


if int(
    s.eq(99999).sum()
) != 0:
    raise ValueError(
        "Unexpected OCD270=99999."
    )


if int(
    s.isna().sum()
) != 3075:
    raise ValueError(
        "OCD270 missing count differs."
    )


print(
    "\nPASS: OCD270 matches CDC."
)


# ============================================================
# 15. CDC CHECK - OCQ290Q
# ============================================================

#
# Hours/day exposed to tobacco smoke
#
# 0-24   = 599
# 7777   = 0
# 9999   = 2
# Missing = 5,945
#
# Attached source zero count = 34
#

s = df["OCQ290Q"]


if int(
    s.between(
        0,
        24
    ).sum()
) != 599:
    raise ValueError(
        "OCQ290Q 0-24 count differs."
    )


if int(
    s.eq(0).sum()
) != 34:
    raise ValueError(
        "OCQ290Q zero count differs."
    )


if int(
    s.eq(7777).sum()
) != 0:
    raise ValueError(
        "Unexpected OCQ290Q=7777."
    )


if int(
    s.eq(9999).sum()
) != 2:
    raise ValueError(
        "OCQ290Q=9999 count differs."
    )


if int(
    s.isna().sum()
) != 5945:
    raise ValueError(
        "OCQ290Q missing count differs."
    )


print(
    "PASS: OCQ290Q matches CDC."
)


# ============================================================
# 16. CDC CHECK - OCD395
# ============================================================

#
# Duration longest job in months
#
# 1-720  = 3,708
# 0      = 15
# 77777  = 0
# 99999  = 5
# Missing = 2,818
#

s = df["OCD395"]


if int(
    s.between(
        1,
        720
    ).sum()
) != 3708:
    raise ValueError(
        "OCD395 1-720 count differs."
    )


if int(
    s.eq(0).sum()
) != 15:
    raise ValueError(
        "OCD395 zero count differs."
    )


if int(
    s.eq(77777).sum()
) != 0:
    raise ValueError(
        "Unexpected OCD395=77777."
    )


if int(
    s.eq(99999).sum()
) != 5:
    raise ValueError(
        "OCD395=99999 count differs."
    )


if int(
    s.isna().sum()
) != 2818:
    raise ValueError(
        "OCD395 missing count differs."
    )


print(
    "PASS: OCD395 matches CDC."
)


# ============================================================
# 17. CDC CHECK - WORKPLACE EXPOSURE DURATION FIELDS
# ============================================================

EXPOSURE_CHECKS = {

    "OCQ520": {
        "low": 0,
        "high": 60,
        "range_count": 1801,
        "zero_count": 147,
        "refused_code": 7777,
        "refused_count": 0,
        "dk_code": 9999,
        "dk_count": 7,
        "missing": 4738
    },

    "OCQ540": {
        "low": 0,
        "high": 67,
        "range_count": 1243,
        "zero_count": 99,
        "refused_code": 7777,
        "refused_count": 1,
        "dk_code": 9999,
        "dk_count": 5,
        "missing": 5297
    },

    "OCQ560": {
        "low": 0,
        "high": 60,
        "range_count": 1394,
        "zero_count": 82,
        "refused_code": 7777,
        "refused_count": 0,
        "dk_code": 9999,
        "dk_count": 6,
        "missing": 5146
    },

    "OCQ580": {
        "low": 0,
        "high": 60,
        "range_count": 1761,
        "zero_count": 142,
        "refused_code": 7777,
        "refused_count": 0,
        "dk_code": 9999,
        "dk_count": 6,
        "missing": 4779
    }
}


for col, cfg in (
    EXPOSURE_CHECKS.items()
):

    s = df[col]


    if int(
        s.between(
            cfg["low"],
            cfg["high"]
        ).sum()
    ) != cfg["range_count"]:
        raise ValueError(
            f"{col}: CDC range count differs."
        )


    if int(
        s.eq(0).sum()
    ) != cfg["zero_count"]:
        raise ValueError(
            f"{col}: zero count differs."
        )


    if int(
        s.eq(
            cfg["refused_code"]
        ).sum()
    ) != cfg["refused_count"]:
        raise ValueError(
            f"{col}: refused count differs."
        )


    if int(
        s.eq(
            cfg["dk_code"]
        ).sum()
    ) != cfg["dk_count"]:
        raise ValueError(
            f"{col}: don't-know count differs."
        )


    if int(
        s.isna().sum()
    ) != cfg["missing"]:
        raise ValueError(
            f"{col}: missing count differs."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 18. CDC ANCHOR CHECK - OCD150
# ============================================================

EXPECTED_OCD150 = {
    1: 3338,
    2: 133,
    3: 142,
    4: 2931,
    7: 2
}


actual = (
    df["OCD150"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OCD150:
    raise ValueError(
        "OCD150 frequency mismatch."
    )


if int(
    df["OCD150"]
    .isna()
    .sum()
) != 0:
    raise ValueError(
        "OCD150 missing count differs."
    )


print(
    "\nPASS: OCD150 matches CDC."
)


# ============================================================
# 19. CDC ANCHOR CHECK - OCQ180
# ============================================================

s = df["OCQ180"]


if int(
    s.between(
        1,
        123
    ).sum()
) != 3334:
    raise ValueError(
        "OCQ180 1-123 count differs."
    )


if int(
    s.eq(77777).sum()
) != 0:
    raise ValueError(
        "Unexpected OCQ180=77777."
    )


if int(
    s.eq(99999).sum()
) != 4:
    raise ValueError(
        "OCQ180=99999 count differs."
    )


if int(
    s.isna().sum()
) != 3208:
    raise ValueError(
        "OCQ180 missing count differs."
    )


print(
    "PASS: OCQ180 matches CDC."
)


# ============================================================
# 20. CDC ANCHOR CHECK - OCQ260
# ============================================================

EXPECTED_OCQ260 = {
    1: 2591,
    2: 81,
    3: 198,
    4: 219,
    5: 362,
    6: 10,
    77: 1,
    99: 9
}


actual = (
    df["OCQ260"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OCQ260:
    raise ValueError(
        "OCQ260 frequency mismatch."
    )


if int(
    df["OCQ260"]
    .isna()
    .sum()
) != 3075:
    raise ValueError(
        "OCQ260 missing count differs."
    )


print(
    "PASS: OCQ260 matches CDC."
)


# ============================================================
# 21. BUILD EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# expected_df =
#
# original released XPT
#
# +
# ONLY the seven validated tiny -> zero corrections.
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 22. VERIFY ALL VALUES ARE WHOLE BEFORE INT64
# ============================================================

for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ):
        raise ValueError(
            f"STOP: {col} contains a non-whole value."
        )


print(
    "\nPASS: All 24 OCQ_E fields contain "
    "whole-number values after correction."
)


# ============================================================
# 23. CLEAN INTEGER-STYLE FORMATTING
# ============================================================

#
# Examples:
#
# SEQN:
# 41475.0 -> 41475
#
# OCD270:
# tiny -> 0
#
# OCQ290Q:
# tiny -> 0
#
# OCQ520:
# tiny -> 0
#
# Special codes:
# 99999.0 -> 99999
#
# Missing:
# NaN -> blank
#

for col in EXPECTED_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All OCQ_E fields converted "
    "to clean integer-style formatting."
)


# ============================================================
# 24. PRE-EXPORT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):
        pre_export_failures.append(
            col
        )


if pre_export_failures:
    raise ValueError(
        "Unexpected pre-export changes:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: Pre-export values equal expected XPT "
    "after ONLY seven validated zero restorations."
)


# ============================================================
# 25. EXPORT CSV
# ============================================================

#
# NO:
#
# round()
# float_format
# recoding
# missing-value filling
#
# Missing values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 26. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 27. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:
    raise ValueError(
        "CSV shape differs from XPT."
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 28. EXACT NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:
        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:
    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 24 columns numerically match "
    "expected XPT values exactly."
)


# ============================================================
# 29. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:
        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:
    raise ValueError(
        "Missing-position fidelity failure:\n"
        + str(missing_failures)
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 30. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 6546:
    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 31. STRICT RAW TOKEN FIDELITY
# ============================================================

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    for example in raw_examples:
        print(example)


    raise ValueError(
        "OCQ_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected XPT-derived representation."
)


# ============================================================
# 32. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in EXPECTED_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Unwanted '.0' values:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 33. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 34. TINY ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:
    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains."
)


# ============================================================
# 35. FINAL ZERO RECHECK
# ============================================================

for col, expected_count in (
    EXPECTED_TINY_COUNTS.items()
):

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:
        raise ValueError(
            f"Final {col} zero count changed."
        )


final_total_zeros = int(
    csv_df[
        EXPECTED_COLUMNS
    ]
    .eq(0)
    .sum()
    .sum()
)


if final_total_zeros != 599:
    raise ValueError(
        "Unexpected final total-zero count."
    )


print(
    "PASS: Final zero counts remain exact."
)


# ============================================================
# 36. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("OCQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "OCD270   = 80"
)

print(
    "OCQ290Q  = 34"
)

print(
    "OCD395   = 15"
)

print(
    "OCQ520   = 147"
)

print(
    "OCQ540   = 99"
)

print(
    "OCQ560   = 82"
)

print(
    "OCQ580   = 142"
)

print(
    "TOTAL    = 599"
)


print(
    "\nGenuine decimal variables:",
    0
)


print(
    "\nCDC CHECK ITEMS not added:"
)

print(
    "OCQ200"
)

print(
    "OCQ500"
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "OCQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 OCQ_E XPT -> CSV
Rows: 6,546
Columns: 24
PASS: OCQ_E structure = 6,546 rows x 24 columns.
PASS: OCQ200 and OCQ500 correctly absent from released XPT.
PASS: All 24 released fields are numeric.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 599

Tiny values by variable:
OCQ520     147
OCQ580     142
OCQ540      99
OCQ560      82
OCD270      80
OCQ290Q     34
OCD395      15

PASS: Exact tiny pattern confirmed.
PASS: Tiny values occur only in CDC zero-permitted fields.

--- VALIDATED ZERO RESTORATION ---
OCD270    : 80 tiny -> 0
OCQ290Q   : 34 tiny -> 0
OCD395    : 15 tiny -> 0
OCQ520    : 147 tiny -> 0
OCQ540    : 99 tiny -> 0
OCQ560    : 82 tiny -> 0
OCQ580    : 142 tiny -> 0
PASS: All 599 validated zeros restored.
PASS: Final zero counts exact.
PASS: No unexpected zeros introduced.

Remaining tiny artifacts: 0
PASS: No tiny XPORT artifacts remain.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL V